In [ ]:

from keras.datasets import reuters
(x_train, y_train), (x_test, y_test) = reuters.load_data(num_words=10000, test_split=0.3, seed=42)

print(f"X Train Length: {len(x_train)}")
print(f"Y Train Length: {len(y_train)}")
print(f"X Test Length: {len(x_test)}")
print(f"Y Test Length: {len(y_test)}")

print(f"Total data examples: {len(x_train) + len(x_test)}")
print(x_train[0])
print(y_train[0])

vocabulary = reuters.get_word_index()
vocabulary = {v:k  for k, v in vocabulary.items()}

for k,v in vocabulary.items():
    print(k,v)

for word_idx in x_train[0]:
    print(vocabulary.get(word_idx - 3 , '?'))

import numpy as np

# First create an empty numpy matrix full of 0.
# Shape will be the number of rows of data we have, and at each row we will have a 10,000 column vector.
x_train_enc = np.zeros( shape=(len(x_train),10000) )
print(x_train_enc.shape)

# E.g 7859, 10000
# Each row has a 10K dimension vector stored. Each index of that 10K vector represent a word index.

for row_number, word_idx_seq in enumerate(x_train):
    # Wherever we have a word_index, set the appropriate index in the 10K vector to 1
    # E.g [ 1, 39, 566 ] would set index 1, 39, and 566 of our 10K vector to 1. All else would be 0.
    x_train_enc[ row_number, word_idx_seq ] = 1
    
print(x_train_enc[0])


### Same for the x_test set now. Yes, we could make a function for this.

x_test_enc = np.zeros( shape=(len(x_test),10000) )
for row_number, word_idx_seq in enumerate(x_test):
    x_test_enc[ row_number, word_idx_seq ] = 1
    
print(x_test_enc[0])

from keras.utils import to_categorical
y_train_enc = to_categorical(y_train)
y_test_enc = to_categorical(y_test)

print(y_train_enc.shape)

# Model Definition
import tensorflow as tf
import keras
from keras import layers

model = keras.Sequential()
model.add(layers.InputLayer(input_shape=(11,))) # 11 Columns of input
model.add(layers.Dense(32, activation="relu"))
model.add(layers.Dense(1, activation="sigmoid")) # 0->1 floating
model.summary()

model = keras.Sequential()
model.add(layers.InputLayer(input_shape=(10000,))) # 10000 dimensional input.
model.add(layers.Dense(64, activation="relu"))
model.add(layers.Dense(64, activation="relu")) # 2 Hidden Layers
model.add(layers.Dense(46, activation="softmax")) # 46 Output topics possible.
model.summary()

from keras.optimizers import SGD

model.compile(
    loss="categorical_crossentropy",
    optimizer=SGD(learning_rate=0.05),
    metrics=["accuracy"]
)

# Training the model
# Slice notation
x_val = x_train_enc[:1000] # Grab from 0 -> 1000
y_val = y_train_enc[:1000]

x_train_enc_rest = x_train_enc[1000:] # From 1000 -> end
y_train_enc_rest = y_train_enc[1000:]

model_training_history = model.fit(
    x_train_enc_rest,
    y_train_enc_rest,
    batch_size = 512,
    epochs = 120,
    validation_data = (x_val, y_val)
)

# Evaluation
import matplotlib.pyplot as plt

# If we want to be fancy, we can set a theme by uncommenting the below line.
#plt.style.use('ggplot')

# No subplots here, as we're just drawing over the same plot with multiple data.

# Loss and Val Loss metrics.
loss = model_training_history.history['loss']
val_loss = model_training_history.history['val_loss']

# Training loss, and Validation loss. Providing labels is useful for our legend later.
plt.plot(loss, label="Training Loss")
plt.plot(val_loss, label="Validation Loss")

# X and Y axes labels.
plt.ylabel('Loss')
plt.xlabel('epochs')

# Draw a red vertical line at x=60.
plt.axvline(x=60, color='red')

# Display a legend.
plt.legend()

# Predicting test set
### ### TODO:
# Re-create, re-compile, re-train the NN model, setting epochs = 40.

### ### HERE

pred_y = model.predict(x_test_enc)

# Let's look at the first prediction output. To get a feeling for the shape, and values.
print(pred_y[0].shape)
print(pred_y[0])
print(np.sum(pred_y[0])) # Softmax output. Should be approx = 1.

print(np.argmax(pred_y[0])) # Which label is the highest probability

# Information bottleneck. Let's try a smaller model, with only 4 neurons in the hidden layers.
model2 = keras.Sequential()
model2.add(layers.InputLayer(input_shape=(10000,))) # 10000 dimensional input.
model2.add(layers.Dense(4, activation="relu"))
model2.add(layers.Dense(4, activation="relu")) # 2 Hidden Layers
model2.add(layers.Dense(46, activation="softmax")) # 46 Output topics possible.
model2.summary()

model2.compile(
    loss='categorical_crossentropy',
    optimizer=SGD(learning_rate=0.05), # Higher learning rate for demonstration.
    metrics=['accuracy']
)

# Using train, test, val datasets from before.
model2_training_history = model2.fit(
    x_train_enc_rest,
    y_train_enc_rest,
    batch_size = 512,
    epochs = 120,
    validation_data = (x_val, y_val)
)

plt.plot(model2_training_history.history['loss'], label="Training Loss")
plt.plot(model2_training_history.history['val_loss'], label="Validation Loss")
plt.ylabel('Loss')
plt.xlabel('Epochs')
plt.legend()

# Dropout 20%
model3 = keras.Sequential()
model3.add(layers.InputLayer(input_shape=(10000,))) # 10000 dimensional input.

model3.add(layers.Dense(64, activation="relu"))
model3.add(layers.Dropout(0.2))

model3.add(layers.Dense(64, activation="relu")) # 2 Hidden Layers
model3.add(layers.Dropout(0.2))

model3.add(layers.Dense(46, activation="softmax")) # 46 Output topics possible.
model3.summary()

model3.compile(
    loss='categorical_crossentropy',
    optimizer=SGD(learning_rate=0.05), # Higher learning rate for demonstration.
    metrics=['accuracy']
)

# Using train, test, val datasets from before.
model3_training_history = model3.fit(
    x_train_enc_rest,
    y_train_enc_rest,
    batch_size = 512,
    epochs = 240, # train for a bit longer to get a better picture.
    validation_data = (x_val, y_val)
)

# Dropout 50%
model4 = keras.Sequential
model4.add(layers.Dense(64, activation="relu"))
model4.add(layers.Dropout(0.5))

model4.add(layers.Dense(64, activation="relu")) # 2 Hidden Layers
model4.add(layers.Dropout(0.5))

model4.add(layers.Dense(46, activation="softmax")) # 46 Output topics possible.
model4.summary()

model4.compile(
    loss='categorical_crossentropy',
    optimizer=SGD(learning_rate=0.05), # Higher learning rate for demonstration.
    metrics=['accuracy']
)

# Using train, test, val datasets from before.
model4_training_history = model4.fit(
    x_train_enc_rest,
    y_train_enc_rest,
    batch_size = 512,
    epochs = 240, # train for a bit longer to get a better picture.
    validation_data = (x_val, y_val)
)

# Evaluate graphs
#Loss
plt.plot(model3_training_history.history['loss'], '--', label="D0.2 - Train Loss")
plt.plot(model3_training_history.history['val_loss'], label="D0.2 - Val Loss")

plt.plot(model4_training_history.history['loss'], '--', label="D0.5 - Train Loss")
plt.plot(model4_training_history.history['val_loss'], label="D0.5 - Val Loss")

plt.plot(model_training_history.history['loss'], '--', label="Original Train Loss")
plt.plot(model_training_history.history['val_loss'], label="Original Val Loss")

plt.ylabel('Loss')
plt.xlabel('epochs')
plt.legend()

# Accuracy
plt.plot(model3_training_history.history['accuracy'], '--', label="D0.2 - Train Acc")
plt.plot(model4_training_history.history['accuracy'], '-', label="D0.5 - Train Acc")
plt.plot(model_training_history.history['accuracy'], '-.', label="Original Train Acc")

plt.ylabel('Acc')
plt.xlabel('epochs')
plt.legend()

# batch Normalisation
bn_model = keras.Sequential()
bn_model.add(layers.InputLayer(input_shape=(10000,))) # 10000 dimensional input.

bn_model.add(layers.Dense(64))
bn_model.add(layers.BatchNormalization())
bn_model.add(layers.ReLU())

bn_model.add(layers.Dense(64)) # 2 Hidden Layers
bn_model.add(layers.BatchNormalization())
bn_model.add(layers.ReLU())

bn_model.add(layers.Dense(46, activation="softmax")) # 46 Output topics possible.
bn_model.summary()

bn_model.compile(
    loss='categorical_crossentropy',
    optimizer=SGD(learning_rate=0.05), # Higher learning rate for demonstration.
    metrics=['accuracy']
)

# Using train, test, val datasets from before.
bn_model_training_history = bn_model.fit(
    x_train_enc_rest,
    y_train_enc_rest,
    batch_size = 512,
    epochs = 150, # train for a bit longer to get a better picture.
    validation_data = (x_val, y_val)
)
plt.plot(bn_model_training_history.history['loss'], '--', label="BN Train Loss")
plt.plot(bn_model_training_history.history['val_loss'], label="BN Val Loss")

plt.ylabel('Loss')
plt.xlabel('epochs')
plt.legend()

# Observations
# L1 and L2 Regularization
bn_l2_model = keras.Sequential()
bn_l2_model.add(layers.InputLayer(input_shape=(10000,))) # 10000 dimensional input.

bn_l2_model.add(layers.Dense(64, kernel_regularizer='l2'))
bn_l2_model.add(layers.BatchNormalization())
bn_l2_model.add(layers.ReLU())

bn_l2_model.add(layers.Dense(64, kernel_regularizer='l2')) # 2 Hidden Layers
bn_l2_model.add(layers.BatchNormalization())
bn_l2_model.add(layers.ReLU())

bn_l2_model.add(layers.Dense(46, activation="softmax")) # 46 Output topics possible.
bn_l2_model.summary()

bn_l2_model.compile(
    loss='categorical_crossentropy',
    optimizer=SGD(learning_rate=0.05), # Higher learning rate for demonstration.
    metrics=['accuracy']
)

# Using train, test, val datasets from before.
bn_l2_model_training_history = bn_l2_model.fit(
    x_train_enc_rest,
    y_train_enc_rest,
    batch_size = 512,
    epochs = 150, # train for a bit longer to get a better picture.
    validation_data = (x_val, y_val)
)

plt.plot(bn_model_training_history.history['loss'], '--', label="BN Train Loss")
plt.plot(bn_model_training_history.history['val_loss'], label="BN Val Loss")

plt.plot(bn_l2_model_training_history.history['loss'], '--', label="BN L2 Train Loss")
plt.plot(bn_l2_model_training_history.history['val_loss'], label="BN L2 Val Loss")

plt.ylabel('Loss')
plt.xlabel('Epochs')
plt.legend()

X Train Length: 7859
Y Train Length: 7859
X Test Length: 3369
Y Test Length: 3369
Total data examples: 11228
[1, 39, 566, 11, 14, 841, 11, 29, 53, 617, 187, 193, 15, 14, 134, 533, 15, 53, 175, 3758, 948, 15, 14, 4313, 279, 15, 39, 4074, 11, 14, 3814, 11, 123, 248, 867, 377, 1471, 81, 2, 64, 187, 6164, 2116, 617, 79, 335, 7, 48, 14, 187, 6164, 4656, 7, 105, 324, 27, 69, 12, 18, 730, 1698, 5, 4, 134, 248, 867, 377, 171, 2, 75, 40, 251, 18, 79, 5, 175, 324, 27, 4, 867, 377, 171, 17, 12]
3
10996 mdbl
16260 fawc
12089 degussa
8803 woods
13796 hanging
20672 localized
20673 sation
20675 chanthaburi
10997 refunding
8804 hermann
20676 passsengers
20677 stipulate
8352 heublein
20713 screaming
16261 tcby
185 four
1642 grains
20680 broiler
12090 wooden
1220 wednesday
13797 highveld
7593 duffour
20681 0053
3914 elections
2563 270
3551 271
5113 272
3552 273
3400 274
7975 rudman
3401 276
3478 277
3632 278
4309 279
9381 dormancy
7247 errors
3086 deferred
20683 sptnd
8805 cooking
20684 stratabit
16262 